# XR-1 (Xiaomi-Robotics-1-5B) post-training on the FR5 dataset — RunPod

Post-trains `XiaomiRobotics/Xiaomi-Robotics-1-5B` on `Slifold/fr5-pick-place-lerobot-v2`
with **end-effector actions in XR-1's own representation** (relative Δpose in the
current tool frame, 30 entries = 1 s). Everything model-side is Xiaomi's released
code, launched unmodified except for three env-gated, marker-guarded patches.
Read `xr1_eef/README.md` for the analysis behind every choice here.

Lessons from the pi0 runs, baked in so they cannot recur:

| trap | cell | guard |
|---|---|---|
| stale pod checkout / stale kernel modules | 0 | hard-sync to `origin/main`, drop imported modules, assert symbols |
| stale dataset copy with the old 400 instructions | 3 | always re-sync, assert instructions are shared |
| `/dev/shm` 64 MB kills 8 DataLoader workers | 2 | `file_system` sharing patched into their `train.py`, workers configurable |
| wandb blocking on an interactive login | 1 | `WANDB_MODE` resolved up front, never a prompt |
| format mismatch discovered hours in | 5 | one sample through **their** `JsonDataset` + `CustomCollate` before torchrun |
| a run that outlives its best checkpoint | 6 | `save_interval` checkpoints + push; no val loss exists in their pipeline — the offline gate is the judge |

GPU: one **A100-80**. The full 5B fine-tune needs ~80 GB before activations, so the
default here **freezes the VLM and trains the 0.6 B DiT + projectors**
(`FREEZE_VLM = True`). See the params cell for the multi-GPU alternative.

## 0 · Pod setup — sync our repo, verify it is current

In [ ]:
import importlib, os, subprocess, sys
from pathlib import Path

WORK     = Path(os.environ.get("XR1_WORK", "/workspace"))
REPO_URL = "https://github.com/SreevaatsavB/fairino-fr5-policies.git"
REPO_DIR = WORK / "fairino-fr5-act-pipeline"
BRANCH   = "main"


def _git(*args, check=True):
    r = subprocess.run(["git", "-C", str(REPO_DIR), *args], capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()


if not REPO_DIR.exists():
    r = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"clone failed:\n{r.stderr}")
    print(f"cloned {REPO_URL}")
else:
    # `pull --ff-only` refuses on a dirty tree and the old cell swallowed that, so the
    # notebook LOOKED updated while running week-old code. Hard-sync instead, and say
    # exactly what gets discarded.
    _git("fetch", "origin", BRANCH)
    local, remote = _git("rev-parse", "HEAD"), _git("rev-parse", f"origin/{BRANCH}")
    if local != remote:
        dirty = _git("status", "--porcelain")
        if dirty:
            print("discarding local changes in the pod checkout:")
            for line in dirty.splitlines()[:10]:
                print(f"    {line}")
            _git("reset", "--hard", "HEAD"); _git("clean", "-fd")
        _git("checkout", BRANCH); _git("reset", "--hard", f"origin/{BRANCH}")
        print(f"synced {local[:8]} -> {remote[:8]}")
    else:
        print(f"already at origin/{BRANCH}")

sys.path[:0] = [str(REPO_DIR / "common"), str(REPO_DIR / "xr1_eef")]
os.chdir(REPO_DIR)

# a git sync cannot touch modules this kernel already imported — drop them so the
# next `from fr5_to_xr1 import ...` re-reads from disk (order-independent)
importlib.invalidate_caches()
for _m in ("fr5_to_xr1", "dataset", "lerobot_patches", "proprio"):
    if sys.modules.pop(_m, None) is not None:
        print(f"  dropped stale {_m}")

import fr5_to_xr1 as _fx
for _need in ("main", "rpy_deg_to_rotm", "rotm_to_rpy_deg", "chunk_deltas", "TOOL_FRAME_FIX"):
    assert hasattr(_fx, _need), f"fr5_to_xr1.{_need} missing -> stale checkout; restart the kernel and re-run"
print(f"\nHEAD  {_git('log', '--oneline', '-1')}")
print(f"clean {'yes' if not _git('status', '--porcelain') else 'NO'}")

## 1 · Parameters

Only `MAX_STEPS`, `BATCH_SIZE` and `FREEZE_VLM` normally need touching.

In [ ]:
import getpass, os
from pathlib import Path

# ── credentials ────────────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (hf_..., read on Slifold datasets + write for the push): ").strip()
assert HF_TOKEN.startswith("hf_")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = str(WORK / "hf_cache")            # persistent volume, not the container overlay
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "") or ""   # <- or paste here; empty = offline, never a prompt

# ── what to train ──────────────────────────────────────────────────────────────
HF_DATASET_REPO = "Slifold/fr5-pick-place-lerobot-v2"
XR1_MODEL_REPO  = "XiaomiRobotics/Xiaomi-Robotics-1-5B"
MAX_STEPS       = 5000     # their reference is 10 000 @ batch 48 (= 480k samples, ~0.9 epoch of
                           # our 547k chunk starts). 5 000 first; every save_interval is a checkpoint.
BATCH_SIZE      = 48       # per GPU. If OOM with FREEZE_VLM: 24.
SAVE_INTERVAL   = 1000     # steps between checkpoints (each one is pushable)
FREEZE_VLM      = True     # True  = train the DiT + projectors (0.6 B), fits one A100-80
                           # False = their full recipe (VLM too); needs >=2-4 GPUs with ZeRO,
                           #         or a CPU-offload optimizer. Not one A100-80.
NUM_WORKERS     = 4        # their code hardcodes 8; 4 is safe with file_system sharing
VAL_FRAC, SEED  = 0.05, 42 # SAME held-out 20 episodes as every pi0 run (FR5Dataset.episode_split)

# ── paths ──────────────────────────────────────────────────────────────────────
XR1_DIR   = WORK / "Xiaomi-Robotics-1"          # their repo
XR1_PKG   = XR1_DIR / "xr1"                      # the installable package + configs + scripts
DATA_ROOT = WORK / "dataset"                     # our LeRobot-v2 copy
CONV_DIR  = WORK / "xr1_data"                    # converted json/ + configs/fr5.yaml
CKPT_PT   = XR1_PKG / "pretrained_ckpt" / "model_states.pt"
PROJECT, EXP = "xr1", "fr5_delta_eef"
OUT_DIR   = XR1_PKG / "outputs" / f"project_{PROJECT}" / EXP    # where their trainer writes
PUSH_REPO = "auto"                               # auto -> <you>/fr5-xr1-5b

# wandb: decided NOW, not 20 minutes into training. Their train.py constructs a
# WandbLogger unconditionally; without a key wandb.init would block on stdin.
os.environ["WANDB_MODE"] = "online" if WANDB_API_KEY else "offline"
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print(f"WANDB: ONLINE (key ...{WANDB_API_KEY[-4:]})")
else:
    print("WANDB: OFFLINE — runs land in", XR1_PKG / "wandb", "; sync later with `wandb sync <dir>`")

print(f"steps {MAX_STEPS}  batch {BATCH_SIZE}  freeze_vlm {FREEZE_VLM}  workers {NUM_WORKERS}")
print(f"output -> {OUT_DIR}")

## 2 · Environment — their pinned stack, their repo, our three patches

`torch 2.8.0 / transformers 4.57.1 / deepspeed 0.18.9 / lightning 2.5.3 / decord`.
This **replaces** the pod's torch. Flash-attention 2 is required (their VLM is built
with `attn_implementation="flash_attention_2"`).

In [ ]:
import subprocess, sys, platform
print("python", platform.python_version(), "|", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
      capture_output=True, text=True).stdout.strip())

def sh(cmd, cwd=None, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True)
    if check and r.returncode:
        raise SystemExit(f"command failed ({r.returncode}): {cmd}")

if not XR1_DIR.exists():
    sh(f"git clone --depth 1 https://github.com/XiaomiRobotics/Xiaomi-Robotics-1 {XR1_DIR}")
else:
    sh("git fetch -q origin && git reset -q --hard origin/main", cwd=XR1_DIR)
    print("XR-1 repo reset to origin/main (our patches are re-applied below)")

sh(f"{sys.executable} -m pip install -q torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128")
sh(f"{sys.executable} -m pip install -q -e {XR1_PKG}")           # assets/requirements.txt: transformers==4.57.1 etc.
sh(f"{sys.executable} -m pip install -q ninja")
_py = f"cp{sys.version_info.major}{sys.version_info.minor}"
_wheel = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
          f"flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-{_py}-{_py}-linux_x86_64.whl")
sh(f"{sys.executable} -m pip install -q {_wheel} || {sys.executable} -m pip install -q flash-attn==2.8.3 --no-build-isolation")
sh("apt-get -qq update && apt-get -qq install -y libegl1 libgl1 libgles2 tmux > /dev/null 2>&1 || true", check=False)

In [ ]:
# Import check in a FRESH interpreter (this kernel may hold pre-upgrade modules).
import subprocess, sys
chk = subprocess.run([sys.executable, "-c", """
import torch, transformers, deepspeed, lightning, decord, flash_attn, mibot
from mibot.utils.io import ACTION_DIM, STATE_DIM, recover_action
print('torch', torch.__version__, '| transformers', transformers.__version__, '| deepspeed', deepspeed.__version__,
      '| lightning', lightning.__version__, '| flash_attn', flash_attn.__version__, '| decord', decord.__version__)
assert transformers.__version__ == '4.57.1', transformers.__version__
assert torch.cuda.is_available()
print('mibot importable, ACTION_DIM', ACTION_DIM, 'STATE_DIM', STATE_DIM)
"""], capture_output=True, text=True, cwd=str(XR1_PKG))
print(chk.stdout); 
if chk.returncode:
    raise SystemExit(chk.stderr[-2000:])

### Patches to their code (idempotent, marker-guarded)

1. `tools/train.py` — `torch.multiprocessing.set_sharing_strategy("file_system")`.
   RunPod caps `/dev/shm` at 64 MB; 8 workers × prefetch 4 × batch 48 × 2 images
   through the default `file_descriptor` strategy kills every worker at once.
2. `base_datamodule.py` — `num_workers` from `XR1_NUM_WORKERS` (hardcoded 8).
3. `XR1.py` — `XR1_FREEZE_VLM=1` freezes the VLM after build; `json_dataset.py` —
   `lru_cache(32)` → 512 (32 cached JSONs against our 380 files would thrash).

In [ ]:
import re
from pathlib import Path

def patch(path, old, new, marker):
    p = Path(path); s = p.read_text()
    if marker in s:
        print(f"  already patched: {p.relative_to(XR1_DIR)}"); return
    assert old in s, f"anchor not found in {p} — their code changed; inspect before continuing:\n{old}"
    p.write_text(s.replace(old, new, 1)); print(f"  patched: {p.relative_to(XR1_DIR)}")

patch(XR1_PKG / "tools/train.py",
      "import hydra\n",
      "import hydra\nimport torch.multiprocessing  # fr5-patch: /dev/shm is 64 MB on RunPod\n"
      "torch.multiprocessing.set_sharing_strategy(\"file_system\")\n",
      "fr5-patch: /dev/shm")

patch(XR1_PKG / "mibot/data/datamodule/base_datamodule.py",
      "            num_workers=8,",
      "            num_workers=int(__import__('os').environ.get('XR1_NUM_WORKERS', 8)),  # fr5-patch",
      "XR1_NUM_WORKERS")

patch(XR1_PKG / "mibot/models/VLA/XR1.py",
      "        self._build_model()\n",
      "        self._build_model()\n"
      "        if __import__('os').environ.get('XR1_FREEZE_VLM') == '1':  # fr5-patch: one A100-80 cannot hold the 5B full FT\n"
      "            self.vlm.requires_grad_(False)\n"
      "            print('[fr5-patch] VLM frozen; training DiT + projectors only')\n",
      "XR1_FREEZE_VLM")

patch(XR1_PKG / "mibot/data/datasets/json_dataset.py",
      "    @lru_cache(maxsize=32)",
      "    @lru_cache(maxsize=512)  # fr5-patch: 380 episode files",
      "fr5-patch: 380")

# prove the patches parse
import ast
for f in ("tools/train.py", "mibot/data/datamodule/base_datamodule.py", "mibot/models/VLA/XR1.py", "mibot/data/datasets/json_dataset.py"):
    ast.parse((XR1_PKG / f).read_text())
print("all four patched files parse")

## 3 · Data + weights

In [ ]:
import json
from huggingface_hub import snapshot_download, hf_hub_download
import pyarrow.parquet as pq

# ALWAYS sync — snapshot_download is incremental, and a cached tree from before the
# 2026-07-30 instruction rewrite would silently train on the old 400 strings.
snapshot_download(HF_DATASET_REPO, repo_type="dataset", local_dir=str(DATA_ROOT), token=HF_TOKEN)
_tasks = pq.read_table(DATA_ROOT / "meta/tasks.parquet").to_pandas()
_n_ep = json.loads((DATA_ROOT / "meta/info.json").read_text())["total_episodes"]
print(f"dataset: {_n_ep} episodes, {len(_tasks)} instructions:")
for s in _tasks.task: print("   ", s)
assert len(_tasks) * 2 <= _n_ep, (f"{len(_tasks)} instructions for {_n_ep} episodes: NOT the shared vocabulary. "
                                  f"Delete {DATA_ROOT} and re-run.")
_vids = sorted((DATA_ROOT / "videos").rglob("*.mp4"))
print(f"videos: {len(_vids)} (expect 800: 400 x 2 cameras)")
assert len(_vids) == 2 * _n_ep

# XR-1 5B weights (10.2 GB) -> where their README expects them
CKPT_PT.parent.mkdir(parents=True, exist_ok=True)
if not CKPT_PT.exists():
    p = hf_hub_download(XR1_MODEL_REPO, "model_states.pt", token=HF_TOKEN)
    import shutil; shutil.copy(p, CKPT_PT)
print(f"weights: {CKPT_PT} ({CKPT_PT.stat().st_size/1e9:.1f} GB)")

## 4 · Convert to XR-1 format

Held-out episodes are the **same 20** every pi0 run used (`episode_split(400, 0.05, 42)`),
so the offline gate compares like with like. They are not converted — XR-1's pipeline
has no validation pass; evaluation happens offline afterwards.

In [ ]:
import shutil, sys
from dataset import FR5Dataset
import fr5_to_xr1 as fx

train_eps, val_eps = FR5Dataset.episode_split(_n_ep, VAL_FRAC, SEED)
print(f"train {len(train_eps)} episodes | held out {len(val_eps)}: {sorted(val_eps)}")

if CONV_DIR.exists():
    shutil.rmtree(CONV_DIR)
fx.main([str(DATA_ROOT), "--out", str(CONV_DIR), "--episodes", *map(str, sorted(train_eps))])

# install the data config into their configs/, with our batch size
cfg_src = (CONV_DIR / "configs" / "fr5.yaml").read_text()
cfg_src = cfg_src.replace("batch_size: 48", f"batch_size: {BATCH_SIZE}")
(XR1_PKG / "configs" / "data" / "fr5.yaml").write_text(cfg_src)
_jsons = sorted((CONV_DIR / "json").glob("*.json"))
print(f"\n{len(_jsons)} episode JSONs -> {CONV_DIR/'json'}")
print(f"data config -> {XR1_PKG/'configs'/'data'/'fr5.yaml'}  (batch_size {BATCH_SIZE})")

# held-out ids first, so they survive a decoder failure below
val_path = CONV_DIR / "val_episodes.json"
val_path.write_text(json.dumps(sorted(val_eps)))
print("held-out episode ids saved ->", val_path)

# the videos referenced in the JSON must open with THEIR decoder
import decord
d = json.loads(_jsons[0].read_text())
for view in ("ego", "wrist_left"):
    vp = d["observations"][view][0]["path"]
    vr = decord.VideoReader(vp, num_threads=2)
    fr = vr.get_batch([0, d["num_frames"] - 1]).asnumpy()
    assert len(vr) >= d["num_frames"], f"{view}: video has {len(vr)} frames < num_frames {d['num_frames']}"
    print(f"  decord {view}: {len(vr)} frames, frame shape {fr.shape[1:]}  OK")

## 5 · One sample through THEIR loader before torchrun

`JsonDataset` + `CustomCollate` exactly as training will use them. Catches a format
mismatch in seconds instead of after a 10 GB model load. Also measures tokens per
sample so `MAX_LENGTH` (their per-batch token budget; samples beyond it are silently
**dropped**) is set from data, not guessed.

In [ ]:
import os, sys, yaml, torch, numpy as np
os.environ["XR1_NUM_WORKERS"] = str(NUM_WORKERS)
sys.path.insert(0, str(XR1_PKG))
from mibot.data.datasets.json_dataset import JsonDataset
from mibot.data.collate.custom_collate import CustomCollate

params = yaml.safe_load((XR1_PKG / "configs/data/fr5.yaml").read_text())["data"]["params"]
params["max_steps"] = 1                              # keeps the sample index small for this probe
ds = JsonDataset(params)
smp = ds[0]
assert tuple(smp["action"].shape) == (30, 60) and tuple(smp["state"].shape) == (1, 60), (smp["action"].shape, smp["state"].shape)
assert smp["action_mask"].shape == (30, 60)
imgs = [c for m in smp["messages"] for c in m["content"] if isinstance(c, dict) and c.get("type") == "image"]
print(f"sample 0: action {tuple(smp['action'].shape)}, state {tuple(smp['state'].shape)}, images {len(imgs)}, sizes {[im.size for im in imgs]}")
a = smp["action"].numpy(); s = smp["state"].numpy()
print(f"  normalised action entry 0  pos {a[0,0:3].round(2)} aa {a[0,3:6].round(2)} grip {a[0,6]:.2f}   (right arm {np.abs(a[:,8:15]).max():.1f}, must be 0)")
print(f"  normalised state joints    {s[0,:6].round(2)}  gripper {s[0,7]:.2f}   (all within [-1,1]: {bool((np.abs(s)<=1).all())})")
assert np.abs(a[:, 8:15]).max() == 0 and np.abs(a[:, 16:]).max() == 0, "right-arm/waist/base dims must be zero"

col = CustomCollate()
batch = col([ds[0], ds[len(ds)//2]])
tok = batch["input_ids"].shape[1] // 2
print(f"  tokens per sample ~{tok}  (2 views, prompt, state, 30 action tokens)")
MAX_LENGTH = int(tok * BATCH_SIZE * 1.3)
print(f"  MAX_LENGTH = {MAX_LENGTH}  (their default 20000; below tokens x batch the collate silently DROPS samples)")
assert "pixel_values" in batch and batch["action"].shape == (2, 30, 60)
print("loader OK")

## 6 · Train

Their launcher, unmodified, from `xr1/`. Streams the log here and to `train.log`.
Interrupting the cell kills torchrun; the last `save_interval` checkpoint survives.

There is **no validation loss** in this pipeline. `train/flow_loss` will fall; whether
the policy works is decided offline afterwards, not by that curve.

In [ ]:
import os, subprocess, sys, time, signal
from pathlib import Path

env = dict(os.environ,
    RESOURCE_GPU="1", MAX_LENGTH=str(MAX_LENGTH), XR1_NUM_WORKERS=str(NUM_WORKERS),
    XR1_FREEZE_VLM="1" if FREEZE_VLM else "0", TOKENIZERS_PARALLELISM="false",
    WANDB_MODE=os.environ["WANDB_MODE"], HF_HOME=os.environ["HF_HOME"], HF_TOKEN=HF_TOKEN,
    PYTHONUNBUFFERED="1")
cmd = ["bash", "scripts/train.sh",
       f"trainer.project={PROJECT}", f"trainer.exp_name={EXP}", "trainer.default_root_dir=outputs",
       "data=fr5", "model=posttrain", f"model.params.pretrained={CKPT_PT}",
       f"trainer.max_steps={MAX_STEPS}", f"trainer.save_interval={SAVE_INTERVAL}"]
print("cwd", XR1_PKG); print(" ".join(cmd)); print()

log = open(XR1_PKG / "train.log", "a", buffering=1)
proc = subprocess.Popen(cmd, cwd=str(XR1_PKG), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
t0 = time.time()
try:
    for line in proc.stdout:
        log.write(line)
        # keep the notebook readable: progress, losses, checkpoints, errors
        if any(k in line for k in ("it/s", "s/it", "loss", "Epoch", "Saving", "checkpoint", "Error", "error", "Traceback", "fr5-patch", "Trainable", "Non-trainable", "CUDA out of memory")):
            print(line.rstrip()[:200])
    proc.wait()
except KeyboardInterrupt:
    print("\ninterrupting torchrun ...")
    proc.send_signal(signal.SIGINT); proc.wait(timeout=120)
finally:
    log.close()
print(f"\nexit {proc.returncode} after {(time.time()-t0)/3600:.2f} h  |  log: {XR1_PKG/'train.log'}")
ckpts = sorted(OUT_DIR.glob("*.ckpt"))
print("checkpoints:", [c.name for c in ckpts] or "NONE")

## 7 · Push

Uploads exactly the layout `mibot/server/deploy.py` loads:
`<dir>/config.py` and `<dir>/last.ckpt/checkpoint/mp_rank_00_model_states.pt`
(model weights only — DeepSpeed optimizer shards are skipped). On the robot PC:

```bash
hf download <repo> --local-dir posttrain_fr5
cd Xiaomi-Robotics-1/xr1 && bash scripts/deploy.sh $PWD/../posttrain_fr5 1 1     # server on :10086
python fairino-fr5-act-pipeline/xr1_eef/deploy_fr5_xr1.py --no-robot --task "Pick up each blue block and put it in the brown tray."
```

In [ ]:
from huggingface_hub import HfApi, whoami
api = HfApi(token=HF_TOKEN)
repo = PUSH_REPO if PUSH_REPO != "auto" else f"{whoami(token=HF_TOKEN)['name']}/fr5-xr1-5b"
api.create_repo(repo, private=True, exist_ok=True)

last = OUT_DIR / "last.ckpt"
weights = last / "checkpoint" / "mp_rank_00_model_states.pt"
assert weights.exists(), f"no checkpoint at {weights} — did training reach save_interval?"
files = {"config.py": OUT_DIR / "config.py", "config.yaml": OUT_DIR / "config.yaml",
         "last.ckpt/checkpoint/mp_rank_00_model_states.pt": weights,
         "fr5.yaml": XR1_PKG / "configs/data/fr5.yaml", "train.log": XR1_PKG / "train.log",
         "val_episodes.json": CONV_DIR / "val_episodes.json"}
card = f"""---
license: apache-2.0
tags: [robotics, vla, xiaomi-robotics-1, fairino-fr5]
---
# {repo.split('/')[-1]} — XR-1-5B post-trained on the FR5 pick-and-place set

- action space: XR-1 native — relative Δpose in the current tool frame, 30 entries = 1 s; left-arm slot; 2 views (ego = scene D435i, wrist_left = wrist D405)
- base `{XR1_MODEL_REPO}` · freeze_vlm={FREEZE_VLM} · batch {BATCH_SIZE} · {MAX_STEPS} steps · lr 2e-5→5e-6 cosine (their recipe)
- data `{HF_DATASET_REPO}`, {len(train_eps)} train episodes; held-out ids in `val_episodes.json`
- Euler convention for FR5 poses: extrinsic XYZ (proven, see fairino-fr5-act-pipeline/xr1_eef/README.md §5)
- TOOL_FRAME_FIX was identity — verify the tool-frame orientation before robot time (README §6 risk 1)
- serve: `bash scripts/deploy.sh <this dir> 1 1`
"""
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
for dst, src in files.items():
    if src.exists():
        api.upload_file(path_or_fileobj=str(src), path_in_repo=dst, repo_id=repo, commit_message=f"{dst}")
        print(f"  pushed {dst} ({src.stat().st_size/1e9:.2f} GB)")
print("done ->", f"https://huggingface.co/{repo}")

## 8 · What is and is not covered

**Covered:** environment, their code with three guarded patches, data sync + shared-instruction
assert, conversion with the proven Euler convention, a loader smoke test, training with
checkpoints, push in the exact deploy layout.

**Not covered — do before robot time:**
1. **Tool-frame orientation** (`xr1_eef/README.md` §6 risk 1). XR-1 unifies the EE-frame
   orientation across all its data; the FR5 TCP frame may differ by a fixed rotation.
   `TOOL_FRAME_FIX` in `fr5_to_xr1.py` is the hook; it is identity here.
2. **Offline evaluation.** Their pipeline has none. The held-out 20 episodes are in
   `val_episodes.json`; an XR-1 equivalent of `delta_joint/gate.py` (server in the loop,
   teacher-forced first-step error vs the don't-move baseline, gripper commitment) is the
   next piece of work.
3. `deploy_fr5_xr1.py` has not driven the arm.